In [1]:
# Ячейка 1
import numpy as np
import pandas as pd
from pathlib import Path

In [3]:
# Ячейка 2
# Укажи путь к своему npz-файлу
npz_path = Path("C:\\Users\\1\\YandexDisk\\_Projects\\Trace25\\CalciumData\\5_Traces\\Trace_J01_1D_data.npz")

data = np.load(npz_path, allow_pickle=True)

print("Ключи внутри npz:")
print(data.files)

Ключи внутри npz:
['C', 'component_indices', 'asp', 'reconstructions']


In [4]:
# Ячейка 3
# Посмотреть, что лежит под каждым ключом

for key in data.files:
    arr = data[key]
    print(f"\nКлюч: {key}")
    print(f"Тип: {type(arr)}")
    print(f"Shape: {getattr(arr, 'shape', 'no shape')}")
    print(f"Dtype: {getattr(arr, 'dtype', 'no dtype')}")
    
    try:
        print("Первые элементы:")
        print(arr[:5])
    except:
        print("Не удалось показать arr[:5]")


Ключ: C
Тип: <class 'numpy.ndarray'>
Shape: (703, 30204)
Dtype: float32
Первые элементы:
[[-5.6289907e+00 -5.6289907e+00 -3.1337502e+00 ... -3.8839440e+00
  -3.9618073e+00 -4.0361967e+00]
 [-4.4168229e+00 -4.4168229e+00 -4.4168229e+00 ...  5.7599101e+00
   5.3246808e+00  4.9080672e+00]
 [-4.9471188e+00 -4.9471188e+00 -4.3914986e+00 ... -5.8995199e-01
  -7.7945226e-01 -9.6071070e-01]
 [-3.5601032e+00 -3.5601032e+00 -3.5601032e+00 ... -7.2405338e-03
  -1.7843604e-01 -3.4138203e-01]
 [-6.1626539e+00 -6.1626539e+00 -6.1626539e+00 ...  2.7469679e+01
   2.5980711e+01  2.4557659e+01]]

Ключ: component_indices
Тип: <class 'numpy.ndarray'>
Shape: (703,)
Dtype: int32
Первые элементы:
[0 3 4 5 6]

Ключ: asp
Тип: <class 'numpy.ndarray'>
Shape: (703, 30204)
Dtype: float32
Первые элементы:
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]

Ключ: reconstructions
Тип: <class 'numpy.ndarray'>
Shape: (703, 30204)
Dtype: float32

In [5]:
# Ячейка 4
# Функция для попытки превратить объект в таблицу

def to_dataframe(obj):
    # 1. Уже обычный 2D массив
    if isinstance(obj, np.ndarray):
        if obj.ndim == 2:
            return pd.DataFrame(obj)
        
        # 2. structured array / record array
        if obj.dtype.names is not None:
            return pd.DataFrame(obj)
        
        # 3. массив словарей / объектов
        if obj.dtype == object:
            flat = obj.ravel()
            if len(flat) > 0 and isinstance(flat[0], dict):
                return pd.DataFrame(list(flat))
    
    # 4. dict
    if isinstance(obj, dict):
        return pd.DataFrame(obj)
    
    return None

In [6]:
# Ячейка 5
# Пробуем вытащить таблицы

tables = {}

for key in data.files:
    obj = data[key]
    
    # если это pickled object внутри 0-d массива
    if isinstance(obj, np.ndarray) and obj.shape == () and obj.dtype == object:
        obj = obj.item()
    
    df = to_dataframe(obj)
    
    if df is not None:
        tables[key] = df
        print(f"[OK] {key} -> DataFrame shape {df.shape}")
    else:
        print(f"[SKIP] {key} не удалось автоматически превратить в таблицу")

[OK] C -> DataFrame shape (703, 30204)
[SKIP] component_indices не удалось автоматически превратить в таблицу
[OK] asp -> DataFrame shape (703, 30204)
[OK] reconstructions -> DataFrame shape (703, 30204)


In [ ]:
# Ячейка 6
# Посмотреть первые строки найденных таблиц

for key, df in tables.items():
    print(f"\n=== {key} ===")
    display(df.head())

In [ ]:
# Ячейка 7
# Сохранить все таблицы в csv

out_dir = Path("extracted_tables")
out_dir.mkdir(exist_ok=True)

for key, df in tables.items():
    out_path = out_dir / f"{key}.csv"
    df.to_csv(out_path, index=False)
    print(f"Сохранено: {out_path}")

In [7]:
import numpy as np
import pandas as pd
from pathlib import Path

# Папки
input_dir = Path(r"C:\Users\1\YandexDisk\_Projects\Trace25\CalciumData\5_Traces")
output_dir = Path(r"C:\Users\1\YandexDisk\_Projects\Trace25\CalciumData\6_Traces")
output_dir.mkdir(parents=True, exist_ok=True)

# Все npz-файлы
npz_files = sorted(input_dir.glob("*.npz"))

print(f"Найдено npz-файлов: {len(npz_files)}")

for npz_path in npz_files:
    try:
        data = np.load(npz_path, allow_pickle=True)

        if "C" not in data.files:
            print(f"[SKIP] {npz_path.name}: ключ 'C' не найден")
            continue

        C = data["C"]  # shape: (n_neurons, n_frames)
        C_t = C.T      # shape: (n_frames, n_neurons)

        # Имена столбцов
        col_names = [f"cell_{i+1}" for i in range(C_t.shape[1])]
        df = pd.DataFrame(C_t, columns=col_names)

        # Имя выходного файла:
        # Trace_J01_1D_data.npz -> Trace_J01_1D_traces.csv
        out_name = npz_path.name.replace("_data.npz", "_traces.csv")
        if out_name == npz_path.name:
            out_name = npz_path.stem + "_traces.csv"

        out_path = output_dir / out_name
        df.to_csv(out_path, index=False)

        print(f"[OK] {npz_path.name} -> {out_name} | shape {df.shape}")

    except Exception as e:
        print(f"[ERROR] {npz_path.name}: {e}")

Найдено npz-файлов: 106
[OK] Trace_J01_1D_data.npz -> Trace_J01_1D_traces.csv | shape (30204, 703)
[OK] Trace_J01_2D_data.npz -> Trace_J01_2D_traces.csv | shape (28746, 565)
[OK] Trace_J01_3D_data.npz -> Trace_J01_3D_traces.csv | shape (17753, 510)
[OK] Trace_J01_4D_data.npz -> Trace_J01_4D_traces.csv | shape (8935, 428)
[OK] Trace_J02_1D_data.npz -> Trace_J02_1D_traces.csv | shape (30188, 903)
[OK] Trace_J02_2D_data.npz -> Trace_J02_2D_traces.csv | shape (28746, 896)
[OK] Trace_J02_3D_data.npz -> Trace_J02_3D_traces.csv | shape (17753, 787)
[OK] Trace_J02_4D_data.npz -> Trace_J02_4D_traces.csv | shape (8933, 371)
[OK] Trace_J03_1D_data.npz -> Trace_J03_1D_traces.csv | shape (20038, 179)
[OK] Trace_J03_2D_data.npz -> Trace_J03_2D_traces.csv | shape (19068, 182)
[OK] Trace_J03_3D_data.npz -> Trace_J03_3D_traces.csv | shape (11779, 197)
[OK] Trace_J03_4D_data.npz -> Trace_J03_4D_traces.csv | shape (5929, 101)
[OK] Trace_J05_1D_data.npz -> Trace_J05_1D_traces.csv | shape (20027, 778)
[OK]